## 1. Setup & Konfigurasi Path

In [ ]:
import pandas as pd
import numpy as np
import gc
import time
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve, auc,
                              confusion_matrix, classification_report)

import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)

 ## 2. Load Dataset & Konteks Masalah

In [ ]:
app_train = pd.read_csv('G:/Backup/Project/HCI/Dataset/application_train.csv')
app_test = pd.read_csv('G:/Backup/Project/HCI/Dataset/application_test.csv')

print('application_train shape:', app_train.shape)
print('application_test shape:', app_test.shape)
app_train.head()


In [ ]:
col_desc = pd.read_csv('G:/Backup/Project/HCI/Dataset/HomeCredit_columns_description.csv', encoding='latin1')
print('Jumlah kolom terdeskripsi per tabel:')
print(col_desc['Table'].value_counts())

# Definisi resmi TARGET
print()
print(col_desc[(col_desc['Table']=='application_{train|test}.csv') & (col_desc['Row']=='TARGET')]['Description'].values[0])


## 3. Goal, Objective, dan Metrics

**Goal Bisnis:**
> Membantu Home Credit mengidentifikasi klien yang berisiko mengalami kesulitan pembayaran, agar keputusan persetujuan pinjaman lebih tepat sasaran.mengurangi kerugian akibat gagal bayar, sambil tetap membuka akses kredit bagi klien yang layak (financial inclusion).

**Objective:**
> Membangun model **binary classification** yang memprediksi probabilitas klien mengalami kesulitan pembayaran (`TARGET = 1`), berdasarkan data aplikasi dan riwayat kredit historisnya, untuk digunakan sebagai alat bantu keputusan (decision support) pada proses underwriting.

**Metrics:**

*Metric evaluasi model (technical):*
- **AUC-ROC** (primary).robust terhadap data imbalanced
- **Precision-Recall AUC**.lebih informatif untuk kelas minoritas (defaulter)
- **Recall**.penting karena biaya gagal mendeteksi defaulter (FN) lebih mahal
- **Precision**.menjaga model tidak terlalu banyak menolak klien baik
- **F1-score**.keseimbangan precision-recall

*Metric dampak bisnis:*
- Estimasi jumlah kredit bermasalah yang berhasil dicegah (loss avoided)
- Estimasi opportunity cost dari klien baik yang salah ditolak
- Cost-benefit analysis pada berbagai threshold probabilitas


## 4. Exploratory Data Analysis

### 4.1 Distribusi Target

In [ ]:
print(app_train['TARGET'].value_counts())
print((app_train['TARGET'].value_counts(normalize=True)*100).round(2))

app_train['TARGET'].value_counts().plot(kind='bar', title='Distribusi TARGET')
plt.xlabel('TARGET (0=Lancar, 1=Gagal Bayar)')
plt.ylabel('Jumlah')
plt.show()


Data sangat imbalanced (~92:8). Accuracy akan menyesatkan sebagai metric utama.gunakan AUC-ROC / PR-AUC / Recall.

### 4.2 Anomali



In [ ]:
print(app_train['DAYS_EMPLOYED'].describe())
anom_count = (app_train['DAYS_EMPLOYED']==365243).sum()
print(f'Jumlah baris anomali (365243): {anom_count} ({anom_count/len(app_train)*100:.2f}%)')
print('Rate default pada kelompok anomali ini:', app_train[app_train['DAYS_EMPLOYED']==365243]['TARGET'].mean())


Nilai 365243 kemungkinan besar kode untuk klien pensiunan/tidak bekerja (bukan noise acak). Rate defaultnya jauh lebih rendah. Perlu ditangani sebagai flag terpisah, bukan di-drop.

### 4.3 Korelasi Fitur Numerik dengan TARGET

In [ ]:
corr = app_train.corr(numeric_only=True)['TARGET'].sort_values()
print('Top 10 menurunkan risiko:')
print(corr.head(10))
print()
print('Top 10 meningkatkan risiko:')
print(corr.tail(11)[:-1])


### 4.4 Outlier & Kategori Tidak Valid

In [ ]:
print('Outlier AMT_INCOME_TOTAL, max:', app_train['AMT_INCOME_TOTAL'].max())
print(app_train.nlargest(5,'AMT_INCOME_TOTAL')[['SK_ID_CURR','AMT_INCOME_TOTAL','TARGET']])
print()
print('CODE_GENDER value_counts:')
print(app_train['CODE_GENDER'].value_counts())


### 4.5 Rate Default per Kategori

In [ ]:
print('Rate default per NAME_EDUCATION_TYPE:')
print(app_train.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean().sort_values(ascending=False))
print()
print('Rate default per NAME_FAMILY_STATUS:')
print(app_train.groupby('NAME_FAMILY_STATUS')['TARGET'].mean().sort_values(ascending=False))


Rate default menurun monoton seiring naiknya tingkat pendidikan (Lower secondary 10.9% → Academic degree 1.8%).

### 4.6 Duplikat & Konsistensi Referensial Antar Tabel

In [ ]:
bureau = pd.read_csv('G:/Backup/Project/HCI/Dataset/bureau.csv')
prev = pd.read_csv('G:/Backup/Project/HCI/Dataset/previous_application.csv')

print('Duplikat baris:')
print('application_train:', app_train.duplicated().sum())
print('bureau:', bureau.duplicated().sum())
print('previous_application:', prev.duplicated().sum())

all_curr = set(app_train['SK_ID_CURR']).union(set(app_test['SK_ID_CURR']))
print()
print('Overlap SK_ID_CURR train-test (harus 0):', len(set(app_train['SK_ID_CURR']) & set(app_test['SK_ID_CURR'])))
print('Cakupan klien di bureau:', f"{bureau['SK_ID_CURR'].nunique()}/{len(all_curr)}")
print('Cakupan klien di previous_application:', f"{prev['SK_ID_CURR'].nunique()}/{len(all_curr)}")
del bureau, prev
gc.collect()


Tidak ada duplikat maupun kebocoran data train/test. Sebagian klien tidak punya riwayat di beberapa tabel (misal hanya 29% punya riwayat kartu kredit).ini bermakna 'tidak punya produk tersebut', bukan missing biasa.

## 5. Data Cleaning & Feature Engineering

### 5.1 Cleaning Tabel Utama (`application`)

In [ ]:
def clean_application(df):
    df = df.copy()

    # Anomali DAYS_EMPLOYED (365243 = kode untuk pensiunan/tidak bekerja)
    df['FLAG_DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

    # CODE_GENDER 'XNA' -> ganti dengan modus
    mode_gender = df.loc[df['CODE_GENDER'] != 'XNA', 'CODE_GENDER'].mode()[0]
    df['CODE_GENDER'] = df['CODE_GENDER'].replace('XNA', mode_gender)

    # Outlier AMT_INCOME_TOTAL -> cap di persentil 99.5
    cap = df['AMT_INCOME_TOTAL'].quantile(0.995)
    df['AMT_INCOME_TOTAL_CAPPED'] = df['AMT_INCOME_TOTAL'].clip(upper=cap)

    # Konversi DAYS_* jadi lebih interpretable (positif, dalam tahun)
    for c in ['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']:
        df[c.replace('DAYS_', 'YEARS_')] = -df[c] / 365

    # Fitur rasio finansial
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['DAYS_EMPLOYED_BIRTH_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']

    return df

train_clean = clean_application(app_train)
test_clean = clean_application(app_test)
print('Shape setelah cleaning - train:', train_clean.shape, '| test:', test_clean.shape)


### 5.2 Agregasi `bureau` + `bureau_balance` → per `SK_ID_CURR`

In [ ]:
bb = pd.read_csv('G:/Backup/Project/HCI/Dataset/bureau_balance.csv')
bb['IS_OVERDUE'] = bb['STATUS'].isin(['1','2','3','4','5']).astype(int)
bb['OVERDUE_SEVERITY'] = bb['STATUS'].replace({'C':0,'X':0,'0':0,'1':1,'2':2,'3':3,'4':4,'5':5}).astype(int)

bb_agg = bb.groupby('SK_ID_BUREAU').agg(
    BB_MONTHS_COUNT=('MONTHS_BALANCE','count'),
    BB_MONTHS_MIN=('MONTHS_BALANCE','min'),
    BB_OVERDUE_MONTHS_COUNT=('IS_OVERDUE','sum'),
    BB_OVERDUE_SEVERITY_MAX=('OVERDUE_SEVERITY','max'),
    BB_OVERDUE_SEVERITY_MEAN=('OVERDUE_SEVERITY','mean'),
).reset_index()
bb_agg['BB_OVERDUE_RATIO'] = bb_agg['BB_OVERDUE_MONTHS_COUNT'] / bb_agg['BB_MONTHS_COUNT']
del bb
gc.collect()

bureau = pd.read_csv('G:/Backup/Project/HCI/Dataset/bureau.csv')
bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
del bb_agg
gc.collect()

bureau['CREDIT_OVERDUE_FLAG'] = (bureau['CREDIT_DAY_OVERDUE'] > 0).astype(int)
bureau['CREDIT_DEBT_RATIO'] = bureau['AMT_CREDIT_SUM_DEBT'] / bureau['AMT_CREDIT_SUM'].replace(0, np.nan)

agg_dict = {
    'SK_ID_BUREAU': 'count',
    'DAYS_CREDIT': ['min','max','mean'],
    'CREDIT_DAY_OVERDUE': ['max','mean'],
    'CREDIT_OVERDUE_FLAG': 'sum',
    'AMT_CREDIT_SUM': ['sum','mean','max'],
    'AMT_CREDIT_SUM_DEBT': ['sum','mean'],
    'AMT_CREDIT_SUM_OVERDUE': ['sum','mean','max'],
    'CREDIT_DEBT_RATIO': 'mean',
    'CNT_CREDIT_PROLONG': 'sum',
    'BB_OVERDUE_MONTHS_COUNT': ['sum','mean'],
    'BB_OVERDUE_SEVERITY_MAX': 'max',
    'BB_OVERDUE_RATIO': 'mean',
}
bureau_agg = bureau.groupby('SK_ID_CURR').agg(agg_dict)
bureau_agg.columns = ['BUREAU_' + '_'.join(col).upper() for col in bureau_agg.columns]
bureau_agg = bureau_agg.reset_index()

active_counts = bureau[bureau['CREDIT_ACTIVE']=='Active'].groupby('SK_ID_CURR').size()
bureau_agg = bureau_agg.merge(active_counts.rename('BUREAU_ACTIVE_COUNT'), on='SK_ID_CURR', how='left')
bureau_agg['BUREAU_ACTIVE_COUNT'] = bureau_agg['BUREAU_ACTIVE_COUNT'].fillna(0)
bureau_agg['BUREAU_ACTIVE_RATIO'] = bureau_agg['BUREAU_ACTIVE_COUNT'] / bureau_agg['BUREAU_SK_ID_BUREAU_COUNT']

print('bureau_agg shape:', bureau_agg.shape)
del bureau
gc.collect()


### 5.3 Agregasi `previous_application` → per `SK_ID_CURR`

In [ ]:
prev = pd.read_csv('G:/Backup/Project/HCI/Dataset/previous_application.csv')
prev['APP_CREDIT_RATIO'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT'].replace(0, np.nan)
prev['IS_APPROVED'] = (prev['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
prev['IS_REFUSED'] = (prev['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)

agg_dict = {
    'SK_ID_PREV': 'count',
    'AMT_ANNUITY': ['mean','max'],
    'AMT_APPLICATION': ['mean','max','sum'],
    'AMT_CREDIT': ['mean','max','sum'],
    'AMT_DOWN_PAYMENT': ['mean'],
    'APP_CREDIT_RATIO': 'mean',
    'DAYS_DECISION': ['min','max','mean'],
    'CNT_PAYMENT': ['mean','sum'],
    'IS_APPROVED': 'sum',
    'IS_REFUSED': 'sum',
}
prev_agg = prev.groupby('SK_ID_CURR').agg(agg_dict)
prev_agg.columns = ['PREV_' + '_'.join(col).upper() for col in prev_agg.columns]
prev_agg = prev_agg.reset_index()
prev_agg['PREV_APPROVAL_RATIO'] = prev_agg['PREV_IS_APPROVED_SUM'] / prev_agg['PREV_SK_ID_PREV_COUNT']
prev_agg['PREV_REFUSAL_RATIO'] = prev_agg['PREV_IS_REFUSED_SUM'] / prev_agg['PREV_SK_ID_PREV_COUNT']

print('prev_agg shape:', prev_agg.shape)
del prev
gc.collect()


### 5.4 Agregasi `POS_CASH_balance` → per `SK_ID_CURR`

In [ ]:
pos = pd.read_csv('G:/Backup/Project/HCI/Dataset/POS_CASH_balance.csv')
pos['IS_DPD'] = (pos['SK_DPD'] > 0).astype(int)

agg_dict = {
    'SK_ID_PREV': 'nunique',
    'MONTHS_BALANCE': 'count',
    'CNT_INSTALMENT': 'mean',
    'CNT_INSTALMENT_FUTURE': 'mean',
    'SK_DPD': ['max','mean'],
    'SK_DPD_DEF': ['max','mean'],
    'IS_DPD': 'sum',
}
pos_agg = pos.groupby('SK_ID_CURR').agg(agg_dict)
pos_agg.columns = ['POS_' + '_'.join(col).upper() for col in pos_agg.columns]
pos_agg = pos_agg.reset_index()
pos_agg['POS_DPD_RATIO'] = pos_agg['POS_IS_DPD_SUM'] / pos_agg['POS_MONTHS_BALANCE_COUNT']

print('pos_agg shape:', pos_agg.shape)
del pos
gc.collect()


### 5.5 Agregasi `credit_card_balance` → per `SK_ID_CURR`

In [ ]:
cc = pd.read_csv('G:/Backup/Project/HCI/Dataset/credit_card_balance.csv')
cc['IS_DPD'] = (cc['SK_DPD'] > 0).astype(int)
cc['UTILIZATION'] = cc['AMT_BALANCE'] / cc['AMT_CREDIT_LIMIT_ACTUAL'].replace(0, pd.NA)

agg_dict = {
    'SK_ID_PREV': 'nunique',
    'MONTHS_BALANCE': 'count',
    'AMT_BALANCE': ['mean','max'],
    'AMT_CREDIT_LIMIT_ACTUAL': 'mean',
    'UTILIZATION': 'mean',
    'AMT_DRAWINGS_CURRENT': ['mean','sum'],
    'AMT_PAYMENT_TOTAL_CURRENT': ['mean','sum'],
    'SK_DPD': ['max','mean'],
    'IS_DPD': 'sum',
}
cc_agg = cc.groupby('SK_ID_CURR').agg(agg_dict)
cc_agg.columns = ['CC_' + '_'.join(col).upper() for col in cc_agg.columns]
cc_agg = cc_agg.reset_index()
cc_agg['CC_UTILIZATION_MEAN'] = pd.to_numeric(cc_agg['CC_UTILIZATION_MEAN'], errors='coerce')
cc_agg['CC_DPD_RATIO'] = cc_agg['CC_IS_DPD_SUM'] / cc_agg['CC_MONTHS_BALANCE_COUNT']

print('cc_agg shape:', cc_agg.shape)
del cc
gc.collect()


### 5.6 Agregasi `installments_payments` → per `SK_ID_CURR`

In [ ]:
inst = pd.read_csv('G:/Backup/Project/HCI/Dataset/installments_payments.csv')

inst['DAYS_LATE'] = (inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']).clip(lower=0)
inst['PAYMENT_SHORTFALL'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
inst['IS_LATE'] = (inst['DAYS_LATE'] > 0).astype(int)
inst['PAYMENT_RATIO'] = inst['AMT_PAYMENT'] / inst['AMT_INSTALMENT'].replace(0, pd.NA)

agg_dict = {
    'SK_ID_PREV': 'nunique',
    'NUM_INSTALMENT_NUMBER': 'count',
    'DAYS_LATE': ['max','mean'],
    'IS_LATE': 'sum',
    'PAYMENT_SHORTFALL': ['sum','mean'],
    'PAYMENT_RATIO': 'mean',
    'AMT_INSTALMENT': ['sum','mean'],
    'AMT_PAYMENT': ['sum','mean'],
}
inst_agg = inst.groupby('SK_ID_CURR').agg(agg_dict)
inst_agg.columns = ['INST_' + '_'.join(col).upper() for col in inst_agg.columns]
inst_agg = inst_agg.reset_index()
inst_agg['INST_PAYMENT_RATIO_MEAN'] = pd.to_numeric(inst_agg['INST_PAYMENT_RATIO_MEAN'], errors='coerce')
inst_agg['INST_LATE_RATIO'] = inst_agg['INST_IS_LATE_SUM'] / inst_agg['INST_NUM_INSTALMENT_NUMBER_COUNT']

print('inst_agg shape:', inst_agg.shape)
del inst
gc.collect()


### 5.7 Gabungkan Semua ke Master Dataset

In [ ]:
def merge_all(df):
    df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')
    df = df.merge(prev_agg, on='SK_ID_CURR', how='left')
    df = df.merge(pos_agg, on='SK_ID_CURR', how='left')
    df = df.merge(cc_agg, on='SK_ID_CURR', how='left')
    df = df.merge(inst_agg, on='SK_ID_CURR', how='left')
    return df

train_master = merge_all(train_clean)
test_master = merge_all(test_clean)

print('Master train shape:', train_master.shape)
print('Master test shape:', test_master.shape)

train_master.to_pickle('G:/Backup/Project/HCI/Output/train_master.pkl")
test_master.to_pickle('G:/Backup/Project/HCI/Output/test_master.pkl")


kolom hasil merge yang `NaN` bukan data hilang biasa. Artinya klien memang tidak punya riwayat/produk tersebut. Ini ditangani khusus saat imputasi (diisi 0, bukan median).

## 6. Insight Extraction (Mengacu ke Objective)

Menggali pola yang membedakan klien berisiko vs tidak, menggunakan fitur hasil agregasi tabel riwayat.

In [ ]:
corr = train_master.corr(numeric_only=True)['TARGET'].drop('TARGET').sort_values()
print('Top 15 fitur paling menurunkan risiko:')
print(corr.head(15))
print()
print('Top 15 fitur paling meningkatkan risiko:')
print(corr.tail(15))


### 6.1 Riwayat Penolakan Pinjaman Sebelumnya vs Risiko

In [ ]:
df = train_master.copy()
df['refusal_bin'] = pd.cut(df['PREV_REFUSAL_RATIO'].fillna(-1), bins=[-1.1,-0.1,0,0.3,0.6,1.01],
                             labels=['Tidak ada riwayat','0% ditolak','1-30% ditolak','31-60% ditolak','>60% ditolak'])
print(df.groupby('refusal_bin', observed=True)['TARGET'].agg(['mean','count']))


Pola dose-response yang jelas. Klien yang sering ditolak sebelumnya (>60%) punya rate default 2x lipat (16.2%) dibanding yang tidak pernah ditolak (7.1%).

### 6.2 Riwayat Keterlambatan Cicilan vs Risiko

In [ ]:
df['late_bin'] = pd.cut(df['INST_LATE_RATIO'].fillna(-1), bins=[-1.1,-0.1,0,0.1,0.3,1.01],
                          labels=['Tidak ada riwayat','Tidak pernah telat','1-10% telat','11-30% telat','>30% telat'])
print(df.groupby('late_bin', observed=True)['TARGET'].agg(['mean','count']))


### 6.3 Efek Kombinasi Sinyal Risiko (EXT_SOURCE rendah + riwayat telat bayar)

In [ ]:
df['ext_source_avg'] = df[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1)
low_ext = df['ext_source_avg'] < df['ext_source_avg'].quantile(0.2)
high_late = df['INST_LATE_RATIO'].fillna(0) > 0.1
df['risk_segment'] = np.where(low_ext & high_late, 'Berisiko ganda',
                        np.where(low_ext | high_late, 'Berisiko tunggal', 'Relatif aman'))
print(df.groupby('risk_segment')['TARGET'].agg(['mean','count']))
del df
gc.collect()


Kombinasi sinyal risiko dari sumber berbeda (skor eksternal + perilaku pembayaran internal) menghasilkan segmentasi risiko yang sangat tajam:
- Relatif aman: 4.7% default
- Berisiko tunggal: 11.7% default
- Berisiko ganda: 21.6% default

Ini mendukung pendekatan scorecard/model gabungan multi-sumber, bukan hanya mengandalkan satu jenis data.

## 7. Modeling

### 7.1 Persiapan Data (Encoding & Imputasi)

In [ ]:
y = train_master['TARGET']
train_ids = train_master['SK_ID_CURR']
test_ids = test_master['SK_ID_CURR']
X = train_master.drop(columns=['TARGET','SK_ID_CURR'])
X_test = test_master.drop(columns=['SK_ID_CURR'])

for c in ['CC_UTILIZATION_MEAN', 'INST_PAYMENT_RATIO_MEAN']:
    if c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')
        X_test[c] = pd.to_numeric(X_test[c], errors='coerce')

common_cols = [c for c in X.columns if c in X_test.columns]
X = X[common_cols]
X_test = X_test[common_cols]

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print('Jumlah kolom kategorikal:', len(cat_cols))

X_combined = pd.concat([X, X_test], axis=0, keys=['train','test'])
X_combined = pd.get_dummies(X_combined, columns=cat_cols, dummy_na=True)
X = X_combined.loc['train'].reset_index(drop=True)
X_test = X_combined.loc['test'].reset_index(drop=True)
del X_combined
gc.collect()

print('Shape setelah one-hot encoding - train:', X.shape, '| test:', X_test.shape)

# Imputasi: kolom hasil agregasi -> isi 0 (artinya tidak ada riwayat)
agg_prefixes = ('BUREAU_','PREV_','POS_','CC_','INST_')
agg_cols = [c for c in X.columns if c.startswith(agg_prefixes)]
X[agg_cols] = X[agg_cols].fillna(0)
X_test[agg_cols] = X_test[agg_cols].fillna(0)

# Sisanya -> imputasi median
other_cols = [c for c in X.columns if c not in agg_cols]
medians = X[other_cols].median()
X[other_cols] = X[other_cols].fillna(medians)
X_test[other_cols] = X_test[other_cols].fillna(medians)

print('Sisa missing value:', X.isnull().sum().sum())


### 7.2 Train-Validation Split & Scaling

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)

print('Train shape:', X_train.shape, '| Val shape:', X_val.shape)
print('Proporsi TARGET - train:', y_train.mean().round(4), '| val:', y_val.mean().round(4))


### 7.3 Logistic Regression.Baseline & Hyperparameter Tuning


In [ ]:
# Baseline
baseline = SGDClassifier(loss='log_loss', max_iter=200, random_state=42, n_jobs=1)
baseline.fit(X_train_scaled, y_train)
val_proba_base = baseline.predict_proba(X_val_scaled)[:,1]
print('Baseline Logistic Regression AUC:', round(roc_auc_score(y_val, val_proba_base), 4))


In [ ]:
# Hyperparameter tuning (grid manual atas alpha/regularisasi & class_weight)
param_grid = [
    {'alpha': 0.0001, 'class_weight': None},
    {'alpha': 0.0001, 'class_weight': 'balanced'},
    {'alpha': 0.001, 'class_weight': None},
    {'alpha': 0.001, 'class_weight': 'balanced'},
    {'alpha': 0.01, 'class_weight': 'balanced'},
    {'alpha': 0.00001, 'class_weight': 'balanced'},
]

results = []
best_auc, best_model, best_params = -1, None, None
for params in param_grid:
    model = SGDClassifier(loss='log_loss', max_iter=200, random_state=42,
                           alpha=params['alpha'], class_weight=params['class_weight'], n_jobs=1)
    model.fit(X_train_scaled, y_train)
    proba = model.predict_proba(X_val_scaled)[:,1]
    auc_score = roc_auc_score(y_val, proba)
    results.append({**params, 'auc': auc_score})
    print(f"alpha={params['alpha']}, class_weight={params['class_weight']} -> AUC={auc_score:.4f}")
    if auc_score > best_auc:
        best_auc, best_model, best_params = auc_score, model, params

print()
print('Best params:', best_params, '| Best AUC:', round(best_auc, 4))
val_proba_logreg = best_model.predict_proba(X_val_scaled)[:,1]
print()
print(classification_report(y_val, (val_proba_logreg > 0.5).astype(int)))


### 7.4 Model Pembanding: Random Forest & HistGradientBoosting

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced',
                              random_state=42, n_jobs=1)
rf.fit(X_train, y_train)
val_proba_rf = rf.predict_proba(X_val)[:,1]
print('Random Forest AUC:', round(roc_auc_score(y_val, val_proba_rf), 4))


In [ ]:
hgb = HistGradientBoostingClassifier(max_iter=150, max_depth=6, learning_rate=0.05,
                                       class_weight='balanced', random_state=42)
hgb.fit(X_train, y_train)
val_proba_hgb = hgb.predict_proba(X_val)[:,1]
print('HistGradientBoosting AUC:', round(roc_auc_score(y_val, val_proba_hgb), 4))


## 8. Evaluasi Model

In [ ]:
print('=== Ringkasan ROC-AUC & PR-AUC ===')
for name, proba in [('Logistic Regression (tuned)', val_proba_logreg),
                     ('Random Forest', val_proba_rf),
                     ('HistGradientBoosting', val_proba_hgb)]:
    fpr, tpr, _ = roc_curve(y_val, proba)
    roc_auc_val = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(y_val, proba)
    pr_auc_val = auc(rec, prec)
    print(f'{name}: ROC-AUC={roc_auc_val:.4f} | PR-AUC={pr_auc_val:.4f}')


In [ ]:
plt.figure(figsize=(7,6))
for name, proba in [('Logistic Regression', val_proba_logreg),
                     ('Random Forest', val_proba_rf),
                     ('HistGradientBoosting', val_proba_hgb)]:
    fpr, tpr, _ = roc_curve(y_val, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
plt.plot([0,1],[0,1],'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Perbandingan Model')
plt.legend()
plt.show()


### 8.1 Confusion Matrix pada Berbagai Threshold (HistGradientBoosting)

In [ ]:
for th in [0.3, 0.5, 0.7]:
    pred = (val_proba_hgb > th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    precision = tp/(tp+fp) if (tp+fp)>0 else 0
    recall = tp/(tp+fn) if (tp+fn)>0 else 0
    print(f'Threshold={th}: TN={tn}, FP={fp}, FN={fn}, TP={tp} | Precision={precision:.3f}, Recall={recall:.3f}')


### 8.2 Feature Importance (Koefisien Logistic Regression)

In [ ]:
coefs = pd.Series(best_model.coef_[0], index=X_train.columns)
print('Top 10 fitur meningkatkan risiko:')
print(coefs.sort_values(ascending=False).head(10))
print()
print('Top 10 fitur menurunkan risiko:')
print(coefs.sort_values().head(10))


## 9. Dampak Bisnis & Rekomendasi

Menerjemahkan performa model ke dampak finansial menggunakan `AMT_CREDIT` aktual di data.

- Loss Given Default (LGD) = 45% dari `AMT_CREDIT`
- Margin keuntungan bersih = 10% dari `AMT_CREDIT` untuk klien yang lunas normal

In [ ]:
val_detail = train_master.loc[X_val.index, ['AMT_CREDIT','TARGET']].reset_index(drop=True)

LGD = 0.45
MARGIN = 0.10

net_without_model = (val_detail[val_detail.TARGET==0]['AMT_CREDIT']*MARGIN).sum() - \
                     (val_detail[val_detail.TARGET==1]['AMT_CREDIT']*LGD).sum()

results_biz = []
for th in np.arange(0.05, 0.96, 0.05):
    pred_risky = (val_proba_hgb > th).astype(int)
    d = val_detail.copy()
    d['pred_risky'] = pred_risky
    fn = d[(d.TARGET==1) & (d.pred_risky==0)]
    tn = d[(d.TARGET==0) & (d.pred_risky==0)]
    net = (tn['AMT_CREDIT']*MARGIN).sum() - (fn['AMT_CREDIT']*LGD).sum()
    results_biz.append((round(th,2), net, net - net_without_model))

results_biz_df = pd.DataFrame(results_biz, columns=['threshold','net_outcome','vs_no_model'])
print(results_biz_df.to_string(index=False))

best_row = results_biz_df.loc[results_biz_df['net_outcome'].idxmax()]
print()
print(f"Threshold optimal: {best_row['threshold']} -> Dampak vs no-model: Rp {best_row['vs_no_model']:,.0f}")


**Insight bisnis utama:**
- Threshold agresif (recall tinggi, mis. 0.3) justru **merugikan** secara finansial. Opportunity cost dari menolak klien baik lebih besar dari kerugian yang dihindari
- **Threshold optimal ~0.65** memberi dampak positif terbaik dibanding baseline "approve semua"
- Titik optimal ini sensitif terhadap rasio margin-terhadap-LGD.perlu disesuaikan begitu tim finance memberi angka aktual

**Rekomendasi:**
1. Jangan gunakan threshold default 0.5.mulai dari **~0.65** sebagai titik operasi awal
2. Gunakan model sebagai **tiering/scoring** (auto-approve / review manual / auto-reject), bukan keputusan biner semata
3. **HistGradientBoosting** sebagai model utama (AUC tertinggi), **Logistic Regression** sebagai model pendamping untuk explainability/compliance
